# Trading Strategy Research Analysis

Analyzes experiment results from `results.tsv` and visualizes strategy performance.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

In [ ]:
# Load results
results_path = Path('results.tsv')
if results_path.exists():
    df = pd.read_csv(results_path, sep='\t')
    print(f'Loaded {len(df)} experiment results')
    print(f'Unique strategies: {df["strategy"].nunique()}')
    print(f'Symbols tested: {df["symbol"].unique()}')
    display(df.head(10))
else:
    print('No results.tsv found. Run experiments first.')
    df = pd.DataFrame()

In [ ]:
if len(df) > 0:
    # Experiment outcomes
    status_counts = df['status'].value_counts()
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    
    # Status distribution
    axes[0].pie(status_counts.values, labels=status_counts.index, autopct='%1.1f%%')
    axes[0].set_title('Experiment Outcomes')
    
    # Sharpe distribution
    df['sharpe'].hist(bins=30, ax=axes[1], edgecolor='black')
    axes[1].axvline(x=0, color='red', linestyle='--', label='Zero')
    axes[1].axvline(x=0.5, color='green', linestyle='--', label='Min threshold')
    axes[1].set_title('Sharpe Ratio Distribution')
    axes[1].legend()
    
    # Sharpe over time (running best)
    btc_results = df[df['symbol'] == 'BTCUSDT'].reset_index(drop=True)
    if len(btc_results) > 0:
        btc_results['running_best'] = btc_results['sharpe'].cummax()
        axes[2].plot(btc_results.index, btc_results['sharpe'], 'o-', alpha=0.5, label='Sharpe')
        axes[2].plot(btc_results.index, btc_results['running_best'], 'r-', linewidth=2, label='Running best')
        axes[2].set_title('BTCUSDT Sharpe Over Experiments')
        axes[2].set_xlabel('Experiment #')
        axes[2].legend()
    
    plt.tight_layout()
    plt.savefig('reports/progress.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
if len(df) > 0:
    # Best strategies per symbol
    print('\n=== Best Strategies by Sharpe Ratio ===')
    for symbol in df['symbol'].unique():
        sym_df = df[df['symbol'] == symbol].sort_values('sharpe', ascending=False)
        print(f'\n{symbol}:')
        print(sym_df[['strategy', 'sharpe', 'return_pct', 'max_dd_pct', 'win_rate', 'profit_factor', 'status']].head(5).to_string(index=False))

In [ ]:
if len(df) > 0:
    # Cross-symbol consistency
    print('\n=== Strategy Consistency Across Symbols ===')
    pivot = df.pivot_table(values='sharpe', index='strategy', columns='symbol', aggfunc='last')
    pivot['avg_sharpe'] = pivot.mean(axis=1)
    pivot['min_sharpe'] = pivot[['BTCUSDT', 'ETHUSDT', 'SOLUSDT']].min(axis=1)
    pivot = pivot.sort_values('avg_sharpe', ascending=False)
    print(pivot.head(10).to_string())

In [ ]:
# Run a specific strategy and plot equity curve
from backtest import run_strategy_backtest
from evaluate import compute_metrics

def plot_equity_curve(strategy_path='strategy.py', symbol='BTCUSDT', period='train'):
    result = run_strategy_backtest(strategy_path=strategy_path, symbol=symbol, period=period)
    metrics = compute_metrics(result)
    
    fig, axes = plt.subplots(2, 1, figsize=(14, 8), gridspec_kw={'height_ratios': [3, 1]})
    
    # Equity curve
    axes[0].plot(result.equity_curve, linewidth=1)
    axes[0].set_title(f'{result.strategy_name} | {symbol} | {period} | Sharpe: {metrics["sharpe_ratio"]:.2f}')
    axes[0].set_ylabel('Equity ($)')
    axes[0].axhline(y=result.equity_curve[0], color='gray', linestyle='--', alpha=0.5)
    
    # Drawdown
    peak = np.maximum.accumulate(result.equity_curve)
    dd = (result.equity_curve - peak) / peak * 100
    axes[1].fill_between(range(len(dd)), dd, 0, alpha=0.4, color='red')
    axes[1].set_ylabel('Drawdown (%)')
    axes[1].set_xlabel('Bar')
    
    plt.tight_layout()
    plt.show()
    
    return metrics

# Uncomment to run:
# metrics = plot_equity_curve()